# Tumor annotation and PerturbView guide decoding

This notebook reads completed canonical SpatialData stores, assigns cells to pixel-coordinate tumor GeoJSON polygons, decodes combinatorial FISH measurements from `agg_nuclear_labels`, and exports separate per-slide and cohort AnnData files.

It is intentionally **read-only with respect to SpatialData**: no shapes, tables, transformations, or other elements are written back to a canonical store. Missing cytoplasm aggregation is supported explicitly and is never represented as measured zero.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from spatialdata import read_zarr

from mif_pipeline import load_config
from mif_pipeline.config import get_slide_config
from mif_pipeline.post_analysis import (
    DecodeResult,
    assign_tumor_ids,
    base_raster_shape,
    build_codebook_from_csv,
    build_slide_analysis,
    concat_slide_analyses,
    decode_perturbview,
    export_slide_analysis,
    read_tumor_geojson,
    table_join_diagnostics,
    table_instance_ids,
    table_to_frame,
)

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300})

## Configuration

Use exact existing channel aliases. `ROUND_CHANNELS` order defines the bit index within each round, and dictionary order defines decoding-round order. Tumor thresholds are fit once per slide using tumor-assigned cells with complete finite nuclear measurements. Positions unused by all codebook entries in a round remain visible in raw diagnostics but are excluded from winner and runner-up selection.

In [ ]:
PIPELINE_CONFIG = Path("/path/to/config.yaml")
CODEBOOK_CSV = Path("/path/to/perturbview_codebook.csv")
OUTPUT_ROOT = Path("/path/to/post_analysis")

# Add one entry per completed slide. Metadata columns are copied into analysis.obs.
SLIDES = {
    "SLIDE-XXXX": {
        "geojson": Path("/path/to/SLIDE-XXXX_tumor_regions.geojson"),
        # Optional labels in exact feature order; otherwise properties.name is used.
        "tumor_ids": None,
        "display_channel": "R0_DAPI",
        "metadata": {"sample_id": "sample_001", "condition": "example"},
    },
}

ROUND_CHANNELS = {
    "R1": ["R1_C1", "R1_C2", "R1_C3", "R1_C4"],
    "R2": ["R2_C1", "R2_C2", "R2_C3", "R2_C4"],
    "R3": ["R3_C1", "R3_C2", "R3_C3", "R3_C4"],
}
BITS_PER_ROUND = 4
GUIDE_COLUMN = "base"
BITS_COLUMN = "bits"
# Optional explicit raw-channel alias -> alignment-QC alias association; aliases are never parsed.
MEASUREMENT_TO_ALIGNMENT = {}
RATIO_MIN = 2.0
NULL_QUANTILE = 95.0
SCALING_PERCENTILE = 99.99

DISPLAY_LEVEL = 3
DISPLAY_PERCENTILES = (1.0, 99.9)
MAX_PLOT_CELLS = 200_000
# H5AD already contains analysis.obs; enable only when a full CSV is specifically needed.
WRITE_CELL_ANNOTATIONS_CSV = False
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Visualization helpers

The display image is read lazily from one pyramid level and cropped before materialization. Coordinates and polygon outlines are shown in global microns. Plot subsampling affects visualization only, never tumor assignment, decoding, or export.

In [ ]:
def _level_array(image, level: int, channel_alias: str):
    node = image[f"scale{level}"]
    dataset = node.ds if hasattr(node, "ds") else node
    arrays = list(dataset.data_vars.values()) if hasattr(dataset, "data_vars") else [dataset]
    if len(arrays) != 1:
        raise ValueError(f"Expected one image array at scale{level}; found {len(arrays)}")
    array = arrays[0]
    if "c" in array.dims:
        available = [str(value) for value in array.coords["c"].values]
        if channel_alias not in available:
            raise KeyError(f"Display channel {channel_alias!r} absent from full_image: {available}")
        array = array.sel(c=channel_alias)
    return array.squeeze(drop=True).transpose("y", "x")


def display_crop(sdata, *, channel_alias, pixel_size_um, level=DISPLAY_LEVEL, zoom_um=None):
    image = sdata.images["full_image"]
    base_y, base_x = base_raster_shape(image)
    array = _level_array(image, level, channel_alias)
    level_y, level_x = (int(array.shape[-2]), int(array.shape[-1]))
    resolution_x = pixel_size_um * base_x / level_x
    resolution_y = pixel_size_um * base_y / level_y
    if zoom_um is None:
        x0, x1, y0, y1 = 0.0, base_x * pixel_size_um, 0.0, base_y * pixel_size_um
    else:
        x0, x1, y0, y1 = map(float, zoom_um)
    ix0 = max(0, int(np.floor(x0 / resolution_x)))
    ix1 = min(level_x, int(np.ceil(x1 / resolution_x)))
    iy0 = max(0, int(np.floor(y0 / resolution_y)))
    iy1 = min(level_y, int(np.ceil(y1 / resolution_y)))
    values = np.asarray(array.isel(y=slice(iy0, iy1), x=slice(ix0, ix1)).compute(), dtype=float)
    low, high = np.percentile(values[np.isfinite(values)], DISPLAY_PERCENTILES)
    shown = np.clip((values - low) / max(high - low, 1e-12), 0, 1)
    extent = (ix0 * resolution_x, ix1 * resolution_x, iy1 * resolution_y, iy0 * resolution_y)
    return shown, extent


def _polygon_parts(geometry):
    return [geometry] if geometry.geom_type == "Polygon" else list(geometry.geoms)


def _plot_polygon_outlines(ax, tumors, *, color="cyan"):
    for tumor_id, geometry in zip(tumors.tumor_ids, tumors.global_geometries):
        for polygon in _polygon_parts(geometry):
            x, y = polygon.exterior.xy
            ax.plot(x, y, color=color, linewidth=1.2)
        point = geometry.representative_point()
        ax.text(point.x, point.y, tumor_id, color=color, fontsize=7)


def _plot_subset(frame, *, max_cells=MAX_PLOT_CELLS):
    if len(frame) <= max_cells:
        return frame
    positions = np.linspace(0, len(frame) - 1, max_cells, dtype=int)
    return frame.iloc[positions]


def plot_spatial_labels(
    record, *, column, zoom_um=None, title=None, output_name=None, exclude_values=()
):
    analysis = record["analysis"]
    slide_settings = record["slide_settings"]
    image, extent = display_crop(
        record["sdata"],
        channel_alias=slide_settings["display_channel"],
        pixel_size_um=record["pixel_size_um"],
        zoom_um=zoom_um,
    )
    frame = analysis.obs[[column]].copy()
    frame[["x_um", "y_um"]] = analysis.obsm["spatial"]
    x0, x1, y1, y0 = extent
    frame = frame.loc[
        frame["x_um"].between(x0, x1) & frame["y_um"].between(y0, y1)
    ]
    frame["_plot_label"] = frame[column].fillna("missing").astype(str)
    excluded = {str(value) for value in exclude_values}
    if excluded:
        frame = frame.loc[~frame["_plot_label"].isin(excluded)]
    frame = _plot_subset(frame)
    fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
    ax.imshow(image, cmap="gray", extent=extent, origin="upper", interpolation="nearest")
    categories = sorted(frame["_plot_label"].unique())
    palette = plt.get_cmap("tab20", max(len(categories), 1))
    for index, category in enumerate(categories):
        selected = frame["_plot_label"].eq(category)
        color = "0.65" if category in {"unassigned", "None", "missing"} else palette(index)
        ax.scatter(
            frame.loc[selected, "x_um"], frame.loc[selected, "y_um"],
            s=2, alpha=0.65, color=color, label=category, rasterized=True,
        )
    _plot_polygon_outlines(ax, record["tumors"])
    ax.set(xlim=(x0, x1), ylim=(y1, y0), xlabel="x (µm)", ylabel="y (µm)")
    ax.set_title(title or f"{record['slide_id']}: {column}")
    if len(categories) <= 25:
        ax.legend(markerscale=4, fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
    if output_name:
        path = Path(record["exports"]["h5ad"]).parent / output_name
        fig.savefig(path, dpi=300, bbox_inches="tight")
    return fig, ax


def plot_decode_qc(record):
    analysis = record["analysis"]
    settings = analysis.uns["post_analysis"]["decode_settings"]
    no_call_label = settings.get("no_call_label", "None")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    funnel = record["decode_funnel"]
    rates = funnel[["pass_top", "final_round_pass"]].div(funnel["eligible_cells"], axis=0)
    rates.plot.bar(ax=axes[0], ylim=(0, 1), title="Per-round pass fractions")
    guide_counts = record["guide_counts"].loc[
        record["guide_counts"]["guide"].astype(str).ne(no_call_label)
    ]
    guide_counts.head(25).set_index("guide")["n_cells"].plot.bar(
        ax=axes[1], title="Top guide calls"
    )
    for ax in axes:
        ax.tick_params(axis="x", labelrotation=45)
    path = Path(record["exports"]["h5ad"]).parent / "decode_qc.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    return fig, axes


def compact_decode_metrics(record):
    analysis = record["analysis"]
    obs = analysis.obs
    meta = analysis.uns["post_analysis"]
    settings = meta["decode_settings"]
    no_call_label = settings.get("no_call_label", "None")
    unknown_label = settings.get("unknown_label", "UNK")
    nuclear_complete = ~obs["decode_incomplete_input"].fillna(True).astype(bool)
    eligible = obs["decode_eligible"].fillna(False).astype(bool)
    eligible_obs = obs.loc[eligible]
    calls = obs["decode_guide_call"].astype(str)
    any_call = eligible & calls.ne(no_call_label)
    mapped_call = any_call & calls.ne(unknown_label)
    unknown_call = eligible & calls.eq(unknown_label)
    no_call = eligible & calls.eq(no_call_label)
    valid_bits = set(codebook_frame[BITS_COLUMN].astype(str))
    valid_tuple = eligible_obs["decode_decoded_bits"].astype(str).isin(valid_bits)
    valid_tuple_no_call = valid_tuple & eligible_obs["decode_guide_call"].astype(str).eq(no_call_label)

    def metric(name, count, denominator_name, denominator):
        return {
            "metric": name, "count": int(count), "denominator": denominator_name,
            "fraction": float(count / denominator) if denominator else np.nan,
        }

    overview = pd.DataFrame([
        metric("complete_nuclear_input", nuclear_complete.sum(), "all_master_cells", len(obs)),
        metric("eligible_nuclear_cells", eligible.sum(), "complete_nuclear_input", nuclear_complete.sum()),
        metric("any_barcode_call", any_call.sum(), "eligible_nuclear_cells", eligible.sum()),
        metric("mapped_guide_call", mapped_call.sum(), "eligible_nuclear_cells", eligible.sum()),
        metric("unknown_tuple", unknown_call.sum(), "eligible_nuclear_cells", eligible.sum()),
        metric("eligible_but_no_call", no_call.sum(), "eligible_nuclear_cells", eligible.sum()),
        metric("valid_codebook_tuple", valid_tuple.sum(), "eligible_nuclear_cells", eligible.sum()),
        metric("valid_tuple_but_no_call", valid_tuple_no_call.sum(), "eligible_nuclear_cells", eligible.sum()),
    ]).set_index("metric")

    round_rows = []
    winner_rows = []
    for round_index, (round_name, channels) in enumerate(ROUND_CHANNELS.items()):
        round_rows.append({
            "round": round_name,
            "median_ratio": eligible_obs[f"decode_{round_name}_ratio"].median(),
            "median_top_fold": eligible_obs[f"decode_{round_name}_top_fold"].median(),
            "pass_raw_threshold": eligible_obs[f"decode_{round_name}_pass_top"].mean(),
            "pass_ratio": eligible_obs[f"decode_{round_name}_pass_ratio"].mean(),
            "pass_both": eligible_obs[f"decode_{round_name}_pass"].mean(),
        })
        top = eligible_obs[f"decode_{round_name}_top_idx"]
        used_positions = {values[round_index] for values in codebook}
        for channel_index, channel in enumerate(channels):
            selected = top.eq(channel_index)
            winner_rows.append({
                "round": round_name, "channel": channel,
                "used_in_codebook": channel_index in used_positions,
                "winner_fraction": selected.mean(),
                "pass_fraction_when_winner": eligible_obs.loc[selected, f"decode_{round_name}_pass"].mean() if selected.any() else np.nan,
                "scaling_value": meta["decode_scaling_values"][round_name][channel],
                "raw_threshold": meta["decode_thresholds"][round_name][channel],
            })

    return {
        "overview": overview,
        "round_qc": pd.DataFrame(round_rows).set_index("round"),
        "winner_qc": pd.DataFrame(winner_rows),
    }

## Load and validate the shared codebook

In [ ]:
pipeline_config = load_config(PIPELINE_CONFIG)
codebook_frame, codebook = build_codebook_from_csv(
    CODEBOOK_CSV,
    round_names=list(ROUND_CHANNELS),
    bits_per_round=BITS_PER_ROUND,
    guide_column=GUIDE_COLUMN,
    bits_column=BITS_COLUMN,
)
display(codebook_frame.head())
print(f"Validated {len(codebook_frame):,} unique guide codes")

## Per-slide analysis

Every join below is by the explicit string `instance_id`. Cells in `agg_cell_labels` define the observation universe. Missing nuclear or optional cytoplasm rows become `NaN`; they cannot shift measurements onto a different cell.

In [ ]:
def analyze_slide(slide_id, slide_settings):
    def status(step, message):
        print(f"[{slide_id}] [{step}/6] {message}", flush=True)

    status(1, "Resolving configuration and loading SpatialData")
    slide_config = get_slide_config(pipeline_config, slide_id)
    store_path = Path(slide_config["spatialdata"]["store_path"])
    pixel_size_um = float(slide_config["pixel_size_um"])
    if not store_path.exists():
        raise FileNotFoundError(store_path)
    sdata = read_zarr(store_path)
    if "full_image" not in sdata.images:
        raise KeyError(f"{slide_id} is missing canonical full_image")
    status(1, f"Loaded {store_path}")

    status(2, "Checking table joins and instance IDs")
    diagnostics = table_join_diagnostics(sdata)
    display(diagnostics)
    status(3, f"Loading tumor polygons from {slide_settings['geojson']}")
    tumors = read_tumor_geojson(
        slide_settings["geojson"],
        expected_slide_id=slide_id,
        expected_pixel_size_um=pixel_size_um,
        expected_canvas_shape_yx=base_raster_shape(sdata.images["full_image"]),
        tumor_id_overrides=slide_settings.get("tumor_ids"),
    )
    tumor_ids, tumor_summary = assign_tumor_ids(
        sdata, tumors, progress=lambda message: status(3, message)
    )
    display(tumor_summary)
    n_assigned = int(tumor_ids.astype(str).ne('unassigned').sum())
    status(3, f"Assigned {n_assigned:,}/{len(tumor_ids):,} cells across {len(tumors.tumor_ids):,} tumors")

    status(4, "Preparing nuclear intensities and decoding guides")
    master_ids = table_instance_ids(
        sdata.tables["agg_cell_labels"], table_name="agg_cell_labels"
    )
    nuclear = table_to_frame(
        sdata.tables["agg_nuclear_labels"], table_name="agg_nuclear_labels"
    ).reindex(master_ids)
    tumor_eligible = tumor_ids.reindex(master_ids).astype(str).ne("unassigned")
    decode_result = decode_perturbview(
        nuclear,
        round_channels=ROUND_CHANNELS,
        codebook=codebook,
        tumor_eligible=tumor_eligible,
        ratio_min=RATIO_MIN,
        null_quantile=NULL_QUANTILE,
        scaling_percentile=SCALING_PERCENTILE,
        bits_per_round=BITS_PER_ROUND,
    )
    no_call_label = decode_result.settings["no_call_label"]
    n_called = int(decode_result.cell_calls["decode_guide_call"].ne(no_call_label).sum())
    status(4, f"Produced non-no-call guide results for {n_called:,}/{len(master_ids):,} cells")
    status(5, "Building the integrated per-cell AnnData")
    analysis = build_slide_analysis(
        sdata,
        slide_id=slide_id,
        tumor_ids=tumor_ids,
        decode_result=decode_result,
        nuclear_intensities=nuclear,
        sample_metadata=slide_settings.get("metadata", {}),
    )
    analysis.uns["post_analysis"]["tumor_geojson"] = str(tumors.path)
    analysis.uns["post_analysis"]["tumor_geojson_metadata"] = tumors.metadata
    analysis.uns["post_analysis"]["pipeline_config"] = str(PIPELINE_CONFIG.resolve())
    analysis.uns["post_analysis"]["codebook_csv"] = str(CODEBOOK_CSV.resolve())
    analysis.uns["post_analysis"]["codebook_guide_column"] = GUIDE_COLUMN
    analysis.uns["post_analysis"]["codebook_bits_column"] = BITS_COLUMN
    analysis.uns["post_analysis"]["codebook_guides"] = codebook_frame[GUIDE_COLUMN].astype(str).tolist()
    analysis.uns["post_analysis"]["codebook_bits"] = codebook_frame[BITS_COLUMN].astype(str).tolist()
    missing_measurements = sorted(set(MEASUREMENT_TO_ALIGNMENT) - set(analysis.var_names))
    alignment_columns = set(analysis.uns["post_analysis"]["alignment_columns"])
    missing_alignment = sorted(set(MEASUREMENT_TO_ALIGNMENT.values()) - alignment_columns)
    if missing_measurements or missing_alignment:
        raise KeyError(
            f"Invalid MEASUREMENT_TO_ALIGNMENT aliases: missing measurements={missing_measurements}, "
            f"missing alignment aliases={missing_alignment}"
        )
    analysis.uns["post_analysis"]["measurement_to_alignment"] = dict(MEASUREMENT_TO_ALIGNMENT)
    status(5, f"Built AnnData with {analysis.n_obs:,} cells and {analysis.n_vars:,} intensity channels")
    decode_summary = DecodeResult(
        cell_calls=pd.DataFrame(),
        funnel=decode_result.funnel,
        guide_counts=decode_result.guide_counts,
        thresholds=decode_result.thresholds,
        scaling_values=decode_result.scaling_values,
        settings=decode_result.settings,
    )
    decode_funnel = decode_result.funnel
    guide_counts = decode_result.guide_counts
    del nuclear, decode_result
    status(6, f"Writing derived outputs under {Path(OUTPUT_ROOT) / slide_id}")
    exports = export_slide_analysis(
        analysis,
        slide_id=slide_id,
        output_root=OUTPUT_ROOT,
        tumor_summary=tumor_summary,
        decode_result=decode_summary,
        source_store=store_path,
        tumor_geojson=tumors,
        provenance={
            "pipeline_config": str(PIPELINE_CONFIG.resolve()),
            "codebook_csv": str(CODEBOOK_CSV.resolve()),
            "round_channels": {key: list(value) for key, value in ROUND_CHANNELS.items()},
        },
        write_cell_annotations_csv=WRITE_CELL_ANNOTATIONS_CSV,
        progress=lambda message: status(6, message),
    )
    status(6, f"Complete: {exports['h5ad']}")
    return {
        "slide_id": slide_id, "slide_settings": slide_settings,
        "store_path": store_path, "pixel_size_um": pixel_size_um,
        "sdata": sdata, "tumors": tumors,
        "tumor_summary": tumor_summary, "diagnostics": diagnostics,
        "decode_funnel": decode_funnel, "guide_counts": guide_counts,
        "analysis": analysis, "exports": exports,
    }

In [ ]:
records = {}
for slide_id, slide_settings in SLIDES.items():
    print(f"\n=== {slide_id} ===")
    records[slide_id] = analyze_slide(slide_id, slide_settings)
    print(records[slide_id]["exports"]["h5ad"])

## Inspect guide assignments across slides

Plot called guides for every slide. `None` cells are excluded before visualization subsampling, while the tumor outlines remain overlaid for context. Compact decoding metrics and the two summary plots are displayed slide by slide.

In [ ]:
for slide_id, record in records.items():
    print(f"\n=== {slide_id}: guide inspection ===")
    no_call_label = record["analysis"].uns["post_analysis"]["decode_settings"].get("no_call_label", "None")
    plot_spatial_labels(
        record, column="decode_guide_call", zoom_um=None,
        title=f"{slide_id}: decoded guides", output_name="decoded_guides.png",
        exclude_values={no_call_label},
    )
    plot_decode_qc(record)
    plt.show()
    decode_metrics = compact_decode_metrics(record)
    for metric_name, metric_table in decode_metrics.items():
        print(f"\n{metric_name}")
        display(metric_table)

## Inspect a selected zoom

Set the slide and global-micron crop directly in the cell below. This writes a separate high-resolution guide overlay and does not alter the whole-slide inspection settings.

In [ ]:
ZOOM_SLIDE = "SLIDE-XXXX"
ZOOM_UM = (0.0, 1000.0, 0.0, 1000.0)  # x_min, x_max, y_min, y_max

zoom_record = records[ZOOM_SLIDE]
no_call_label = zoom_record["analysis"].uns["post_analysis"]["decode_settings"].get("no_call_label", "None")
plot_spatial_labels(
    zoom_record, column="decode_guide_call", zoom_um=ZOOM_UM,
    title=f"{ZOOM_SLIDE}: decoded guides (zoom)",
    output_name="decoded_guides_zoom.png",
    exclude_values={no_call_label},
);

## Cohort table

`X` contains whole-cell intensities; the `nucleus` and optional `cytoplasm` layers share that raw-channel axis. Nimbus and alignment ZNCC remain labeled `obsm` matrices because they have their own smaller feature axes. When only some slides have cytoplasm aggregation, missing slide rows are `NaN`, not zero. Concatenation normalizes the derived per-slide AnnData objects in place to avoid deep-copying every slide, then releases those per-slide references once the cohort object exists.

In [ ]:
slide_analyses = {slide_id: record["analysis"] for slide_id, record in records.items()}
cohort = concat_slide_analyses(slide_analyses)
for record in records.values():
    record.pop("analysis")
del slide_analyses
cohort_path = OUTPUT_ROOT / "cohort_cell_analysis.h5ad"
cohort.write_h5ad(cohort_path)
cohort_manifest = {
    "schema_version": 1,
    "slide_order": list(records),
    "n_cells": int(cohort.n_obs),
    "n_intensity_channels": int(cohort.n_vars),
    "cytoplasm_available_by_slide": cohort.uns["post_analysis_cohort"]["cytoplasm_available_by_slide"],
    "nimbus_columns": cohort.uns["post_analysis_cohort"]["nimbus_columns"],
    "alignment_columns": cohort.uns["post_analysis_cohort"]["alignment_columns"],
    "spatial_coordinates_are_slide_local": True,
    "output_h5ad": str(cohort_path.resolve()),
    "source_spatialdata_modified": False,
}
(OUTPUT_ROOT / "cohort_manifest.json").write_text(json.dumps(cohort_manifest, indent=2))
print(cohort)
print(cohort_path.resolve())

## Interpretation and extension

- `decode_guide_call == 'None'` means one or more rounds did not pass; `UNK` means all rounds passed but the tuple was absent from the codebook.
- `obsm['alignment_zncc']` stores correlation only. A residual can be derived as `1 - clip(correlation, 0, 1)` without duplicating the measurement.
- `obsm['spatial']` is in microns but remains slide-local; coordinates from different slides must not be plotted as one shared physical canvas.
- The decoder operates on cell-aggregated nuclear intensities. It is not an individual fluorescent-spot detector.
- If Nimbus or alignment measurements later need their own feature metadata and modeling workflows, promote them to separate MuData modalities rather than padding them into intensity layers.